# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

My lane (Lane 2 — Refresh/Content Opportunity Scoring) is a ranking problem: "which
pages should a reviewer check first?" This needs a score to rank by, not just a
yes/no label, so I evaluate at Precision@K — same as my Week-4 baseline.

I chose Logistic Regression first (readable, shows how much each feature matters)
then Random Forest (usually stronger, can capture nonlinear effects like the
position-tier pattern I found in ML-07). Both output a probability I can rank by,
directly comparable to my baseline's score.

In [11]:
!pip install -q huggingface_hub datasets pandas pyarrow scikit-learn

from huggingface_hub import login
from google.colab import userdata
from datasets import load_dataset
import pandas as pd
import numpy as np

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

MONTH = "2026-03"
ds = load_dataset(
    "FlyRank/internship-warehouse",
    data_files=f"fact_content_daily_performance/month={MONTH}/*.parquet",
    split="train", token=HF_TOKEN,
)
df_month = ds.to_pandas()
df_month["report_date"] = pd.to_datetime(df_month["report_date"])
df_month = df_month[df_month["gsc_data_available"] == True].copy()

first_half = df_month[df_month["report_date"] <= "2026-03-15"]
second_half = df_month[df_month["report_date"] >= "2026-03-16"]
group_cols = ["client_hash_id", "content_hash_id"]

first_agg = first_half.groupby(group_cols).agg(
    impressions_first_half=("gsc_impressions", "sum"),
    clicks_first_half=("gsc_clicks", "sum"),
    avg_position_first_half=("gsc_avg_position", "mean"),
).reset_index()
second_agg = second_half.groupby(group_cols).agg(
    clicks_second_half=("gsc_clicks", "sum"),
).reset_index()

pair_df = first_agg.merge(second_agg, on=group_cols, how="inner")
pair_df["ctr_first_half"] = (pair_df["clicks_first_half"] / pair_df["impressions_first_half"].replace(0, np.nan)) * 100
pair_df["is_declining_proxy"] = (pair_df["clicks_second_half"] < pair_df["clicks_first_half"]).astype(int)
pair_df["position_tier"] = pd.cut(pair_df["avg_position_first_half"], bins=[0,3,10,20,100], labels=["1-3","4-10","11-20","21+"])

# --- Rebuild Week-4 baseline rule (same as ML-07) ---
tier_median_ctr = pair_df.groupby("position_tier", observed=True)["ctr_first_half"].transform("median")
high_volume = (pair_df["impressions_first_half"] >= pair_df["impressions_first_half"].quantile(0.75)).astype(int)
weak_ctr = (pair_df["ctr_first_half"] < tier_median_ctr).astype(int)
pair_df["baseline_score"] = high_volume * weak_ctr * pair_df["impressions_first_half"]

print(f"Total pairs: {len(pair_df):,}")

print("""
METHOD CHOICE:
My lane (Lane 2 — Refresh/Content Opportunity Scoring) is a ranking problem:
"which pages should a reviewer check first?" This needs a SCORE to rank by,
not just a yes/no label — so I evaluate at Precision@K, same as my baseline.

I'm using Logistic Regression first (readable, gives a clean sense of how much
each feature matters) then Random Forest (usually stronger, handles nonlinear
patterns like the position-tier effect I found in ML-07). Both output a
probability I can rank by, directly comparable to my baseline's score.
""")

Total pairs: 141,467

METHOD CHOICE:
My lane (Lane 2 — Refresh/Content Opportunity Scoring) is a ranking problem:
"which pages should a reviewer check first?" This needs a SCORE to rank by,
not just a yes/no label — so I evaluate at Precision@K, same as my baseline.

I'm using Logistic Regression first (readable, gives a clean sense of how much
each feature matters) then Random Forest (usually stronger, handles nonlinear
patterns like the position-tier effect I found in ML-07). Both output a
probability I can rank by, directly comparable to my baseline's score.



## 2. Split design

Split by client (GroupShuffleSplit, 70/30), not by row. The same client_holdout
principle used since ML-01 to ML-04 applies here: if the same client's content
appeared in both train and test, the model could learn client-specific quirks
(a client's overall traffic pattern) rather than genuine page-level signals, and
get an unfair advantage on "unseen" test rows that aren't really unseen. Confirmed
zero client overlap between train (30 clients) and test (13 clients) after the split.

In [12]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped by client: same client should NOT appear in both train and test,
# otherwise the model could learn client-specific quirks and get an unfair
# advantage on "unseen" test rows from the same client — same client_holdout
# principle used throughout ML-01 to ML-04.
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(pair_df, groups=pair_df["client_hash_id"]))

train_df = pair_df.iloc[train_idx].copy()
test_df = pair_df.iloc[test_idx].copy()

print(f"Train: {len(train_df):,} pairs, {train_df['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test_df):,} pairs, {test_df['client_hash_id'].nunique()} clients")

overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print(f"Client overlap between train and test: {len(overlap)} (should be 0)")

Train: 129,698 pairs, 30 clients
Test:  11,769 pairs, 13 clients
Client overlap between train and test: 0 (should be 0)


## 3. Train + compare vs my baseline

Trained Logistic Regression and Random Forest on the same 4 features as my baseline
(impressions_first_half, clicks_first_half, ctr_first_half, avg_position_first_half),
then evaluated all three (baseline rule, LR, RF) on the same held-out test split,
same metric (Precision@10, Precision@20), alongside the test set's base rate.

Result: LR and RF both scored far above the baseline and the base rate — LR hit
Precision@10 = 1.000. This looked too good to be true, so I investigated in Section 4
rather than reporting it at face value.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

features = ["impressions_first_half", "clicks_first_half", "ctr_first_half", "avg_position_first_half"]

X_train = train_df[features].fillna(0)
y_train = train_df["is_declining_proxy"]
X_test = test_df[features].fillna(0)
y_test = test_df["is_declining_proxy"]

lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_train, y_train)

test_df = test_df.copy()
test_df["lr_score"] = lr.predict_proba(X_test)[:, 1]
test_df["rf_score"] = rf.predict_proba(X_test)[:, 1]
# baseline_score already exists in test_df from Section 1

def precision_at_k(df, score_col, k, label_col="is_declining_proxy"):
    return df.sort_values(score_col, ascending=False).head(k)[label_col].mean()

base_rate = test_df["is_declining_proxy"].mean()

results = pd.DataFrame({
    "Method": ["Base rate (random)", "Baseline rule (ML-07)", "Logistic Regression", "Random Forest"],
    "Precision@10": [
        base_rate,
        precision_at_k(test_df, "baseline_score", 10),
        precision_at_k(test_df, "lr_score", 10),
        precision_at_k(test_df, "rf_score", 10),
    ],
    "Precision@20": [
        base_rate,
        precision_at_k(test_df, "baseline_score", 20),
        precision_at_k(test_df, "lr_score", 20),
        precision_at_k(test_df, "rf_score", 20),
    ],
})
print(f"Test set base rate: {base_rate:.3f}\n")
print(results.to_string(index=False))

Test set base rate: 0.248

               Method  Precision@10  Precision@20
   Base rate (random)       0.24777       0.24777
Baseline rule (ML-07)       0.20000       0.10000
  Logistic Regression       1.00000       0.90000
        Random Forest       0.90000       0.95000


## 4. Errors and interpretation

The near-perfect scores (LR: 1.000, RF: 0.90-0.95) are suspiciously high compared to
the base rate (0.248) and are almost certainly NOT genuine "decline prediction" —
investigation confirmed a structural artifact: is_declining_proxy is defined as
(clicks_second_half < clicks_first_half), so whenever clicks_first_half == 0, the
label is mathematically forced to 0 (clicks can't go negative). This gives the model
an easy, mechanical shortcut that has nothing to do with learning real decline
patterns — it's exploiting a boundary condition baked into the label's own formula.

This is a real lesson, not a clean win: a fair comparison needs a label definition
that doesn't structurally depend on the same raw count used as a feature, or the
model should be evaluated only on pairs with clicks_first_half > 0 to remove this
shortcut. I'm reporting the honest, likely-inflated numbers above rather than hiding
them, per the "suspiciously perfect = probably leakage" rule.

Additional limitation: the test set has only 13 clients (grouped split), which is a
small, high-variance sample — precision@K numbers here could shift noticeably with a
different random split.

In [14]:
# --- Feature importance check (RF) ---
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)

# --- Investigate the suspiciously perfect score ---
# is_declining_proxy = (clicks_second_half < clicks_first_half)
# When clicks_first_half == 0, clicks_second_half can NEVER be less than 0 —
# so is_declining_proxy is MATHEMATICALLY FORCED to 0 whenever clicks_first_half == 0.
# This creates an artificial floor effect: a feature used in scoring is also
# structurally tied to how the label itself is defined.

zero_clicks = test_df[test_df["clicks_first_half"] == 0]
print(f"\nRows with clicks_first_half == 0: {len(zero_clicks)}")
print(f"Of those, is_declining_proxy == 0 (i.e. NOT declining): "
      f"{(zero_clicks['is_declining_proxy'] == 0).mean()*100:.1f}% (should be exactly 100%)")

# --- 3 concrete wrong cases ---
test_df["rf_rank"] = test_df["rf_score"].rank(ascending=False)
top10_rf = test_df.sort_values("rf_score", ascending=False).head(10)
wrong = top10_rf[top10_rf["is_declining_proxy"] == 0]
print(f"\nWrong picks in RF's top-10: {len(wrong)}")
print(wrong[["content_hash_id", "clicks_first_half", "clicks_second_half", "rf_score"]].head(3))

Random Forest feature importances:
ctr_first_half             0.492396
clicks_first_half          0.425884
impressions_first_half     0.067047
avg_position_first_half    0.014673
dtype: float64

Rows with clicks_first_half == 0: 6912
Of those, is_declining_proxy == 0 (i.e. NOT declining): 100.0% (should be exactly 100%)

Wrong picks in RF's top-10: 1
                content_hash_id  clicks_first_half  clicks_second_half  \
27336  content_73c45865be5c672d                  1                   1   

       rf_score  
27336  0.868349  


In [15]:
print(wrong[["content_hash_id", "clicks_first_half", "clicks_second_half", "rf_score"]].head(3).to_string())
print(f"\nOnly {len(wrong)} wrong pick(s) found in RF's top-10 — consistent with the")
print("clicks_first_half==0 shortcut explaining most of the apparent 'perfect' performance.")

                content_hash_id  clicks_first_half  clicks_second_half  rf_score
27336  content_73c45865be5c672d                  1                   1  0.868349

Only 1 wrong pick(s) found in RF's top-10 — consistent with the
clicks_first_half==0 shortcut explaining most of the apparent 'perfect' performance.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.